In [ ]:
import torch
from torch import nn
import numpy as np
from tqdm import trange
import matplotlib.pyplot as plt
from IPython.display import clear_output


In [ ]:
N_PITS = 6
N_ACTIONS = N_PITS
BOARD_SIZE = 2*N_PITS+2
N_SEEDS = 4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
from kalaha_env import KalahaEnv
env = KalahaEnv(
    pits_per_player=N_PITS,
    seeds_per_pit=N_SEEDS,
    reward_type='score_delta',
    render_mode='human'
)

In [ ]:
from agent import AgentNetwork, A2CAgent


agent = A2CAgent(
    AgentNetwork(input_size=BOARD_SIZE, N_actions=N_PITS, hidden_size=256).to(device), 
    device=device)

adversary_agent = A2CAgent(
    AgentNetwork(input_size=BOARD_SIZE, N_actions=N_PITS, hidden_size=256).to(device), 
    device=device)

adversary_agent.model.load_state_dict(agent.model.state_dict())

In [ ]:
def compute_a2c_loss(trajectory, entropy_coef = 0.01):
    """
    Compute A2C loss from collected trajectory
    """
    # Convert trajectory to tensors
    value_targets = torch.stack(trajectory['value_targets'], dim=0).to(device)
    values = torch.stack(trajectory['values'], dim=0)
    log_probs = torch.stack(trajectory['log_probs'], dim=0)
    entropy = torch.stack(trajectory['entropy'], dim=0)

    # POLICY LOSS
    advantages = (value_targets - values).detach()
    assert log_probs.requires_grad and not advantages.requires_grad
    policy_loss = -(log_probs*advantages).mean()

    # VALUE LOSS
    assert values.requires_grad and not value_targets.requires_grad
    value_loss = torch.square(values-value_targets.detach()).mean()

    # ENTROPY LOSS
    entropy_loss = entropy_coef * entropy.mean()

    return value_loss + policy_loss + entropy_loss

def compute_n_step_returns(trajectory, last_state_value, gamma=0.99):
    """
    Compute n-step returns in trajectory
    """
    rewards = trajectory["rewards"]
    resets = trajectory["resets"]

    T = len(rewards)

    sm = last_state_value
    value_targets = []
    for t in reversed(range(T)):
        sm = rewards[t] + sm * gamma
        sm = torch.where(torch.tensor(resets[t]), rewards[t], sm)
        value_targets.append(sm)
    trajectory["value_targets"] = value_targets[::-1]

In [ ]:
def generate_session(env, agent, adversary_agent=None, t_max=100000, train=True, epsilon=0.15, n_steps=6):
    
    total_reward = 0
    total_loss = []
    state, info = env.reset()
    terminated = truncated = False
    trajectory ={
        'rewards': [],
        'resets': [],
        'values': [],
        'log_probs': [],
        'entropy' : [],
    }

    for t in range(t_max):
        while not terminated and not truncated and info["current_player"] == 0:
            action_mask = torch.tensor(env.action_masks(), dtype=torch.bool).to(device)
            
            action, log_prob, value, entropy = agent.act(state, action_mask, greedy=False, eps_greedy=True, epsilon=epsilon)
            next_state, reward, terminated, truncated, info = env.step(action)
            reward *= 0.1
            trajectory['rewards'].append(reward)
            trajectory['resets'].append(terminated or truncated)
            trajectory['values'].append(value)
            trajectory['log_probs'].append(log_prob)
            trajectory['entropy'].append(entropy)
            state = next_state
            total_reward += reward
            
        while not terminated and not truncated and info["current_player"] == 1:
            action_mask = torch.tensor(env.action_masks(), dtype=torch.bool).to(device)

            if adversary_agent:
                 action, _, _, _ = adversary_agent.act(state, action_mask, greedy=False, eps_greedy=True, epsilon=.5)
            else: # random
                action = np.random.choice(np.array(info["available_actions"])) 
            state, reward, terminated, truncated, info = env.step(action)
        
        if not (terminated or truncated) and len(trajectory['rewards']) >= n_steps:
            _, _, last_state_value, _  = agent.act(state, action_mask, greedy=False, eps_greedy=True, epsilon=epsilon)
            compute_n_step_returns(trajectory, last_state_value.detach())
            loss = compute_a2c_loss(trajectory)
            trajectory ={
                'rewards': [],
                'resets': [],
                'values': [],
                'log_probs': [],
                'entropy' : [],
            }
            yield loss

            
        # last update
        elif (terminated or truncated):
            trajectory['rewards'][-1] += 50 if info["winner"] == 0 else -50
            last_state_value = 0
            compute_n_step_returns(trajectory, last_state_value)
            loss = compute_a2c_loss(trajectory)
            trajectory ={
                'rewards': [],
                'resets': [],
                'values': [],
                'log_probs': [],
                'entropy' : [],
            }
            return loss

def epsilon_linear_decay(N_steps, eps_max, eps_min, step):
    return max(eps_min, eps_max * (1.0 - step / N_steps))

In [ ]:
@torch.no_grad()
def play(env, agent, t_max=100000, opponent=None, epsilon=.15):
    state, info = env.reset()
    terminated = truncated = False
    for t in range(t_max):
        while not terminated and not truncated and info["current_player"] == 0:
            action_mask = torch.tensor(env.action_masks(), dtype=torch.bool).to(device)
            
            action, _, _,_ = agent.act(state, action_mask, greedy=True, eps_greedy=False)
            _, _, terminated, truncated, info = env.step(action)

        while not terminated and not truncated and info["current_player"] == 1:
            if opponent is None:
                action = np.random.choice(np.array(info["available_actions"])) 
            elif opponent == "Player":
                action = int(input("Action: "))
            else:
                action_mask = torch.tensor(env.action_masks(), dtype=torch.bool).to(device)
                action, _, _, _ = opponent.act(state, action_mask, greedy=False, eps_greedy=False)
            assert action in info["available_actions"], action.__repr__
            _, _, terminated, truncated, info = env.step(action)


        if terminated or truncated:
            return info
    return info

@torch.no_grad()
def eval_agent(env, agent, opponent=None, games=500):
    """
    Evaluate the random agent on the environment.
    """
    c=0
    for _ in trange(games):
        c += play(env, agent, opponent=opponent)["winner"]==0
    return c/games

In [ ]:
def train_agent(agent, adversary_agent=None, N_games=4000, lr=1e-4, eval_steps=400, score_threshold=0.85, eps_max=.5, eps_min=.05):
    opt = torch.optim.Adam(agent.model.parameters(), lr=lr)
    opponent_beaten = False
    trajectory_losses = []
    mean_100_sessions_loss = 0
    games_played = 0
    loss_history = []
    random_winrate_history = []
    selfplay_winrate_history = []

    for k in trange(0, N_games):
        if k % eval_steps == 0:
            vs_opponent_score = eval_agent(env, agent, adversary_agent, games=200)
            vs_random_score = eval_agent(env, agent, games=200)
            opponent_beaten = vs_opponent_score >= score_threshold
            random_winrate_history.append(vs_random_score)
            selfplay_winrate_history.append(vs_opponent_score)
        
        eps = epsilon_linear_decay(N_games, eps_max, eps_min, k)
        if adversary_agent and opponent_beaten:
            print("updating opponent")
            opponent_beaten = False
            adversary_agent.model.load_state_dict(agent.model.state_dict())
        for loss in generate_session(env, agent, adversary_agent, epsilon=eps):
            #torch.nn.utils.clip_grad_norm_(agent.model.parameters(), max_norm=1.0)
            trajectory_losses.append(loss)
            mean_100_sessions_loss += loss.detach().cpu()
        games_played += 1
        if games_played >= 15:  # gradient ascent using batch of 15 games
            games_played = 0
            batch_loss = torch.stack(trajectory_losses).mean()
            batch_loss.backward()
            opt.step()
            opt.zero_grad()
            trajectory_losses = []

        if k > 0 and k % 100 == 0: # update plots
            loss_history.append(mean_100_sessions_loss/100)
            mean_100_sessions_loss = 0.0
            clear_output(True)
            plt.figure(figsize=(18,6))
            plt.subplot(1, 2, 1)
            plt.title('Loss')
            plt.plot(loss_history)
            plt.grid()
            plt.subplot(1, 2, 2)
            plt.title('Winrate')
            plt.plot(random_winrate_history, c='b', label="VS random")
            plt.plot(selfplay_winrate_history, c="r", label="VS ref")
            plt.legend()
            plt.grid()
            plt.show()

In [ ]:
train_agent(agent, adversary_agent=None, N_games=4000, lr=3e-4, eval_steps=400, eps_max=0.5, eps_min=0.15)

In [ ]:
eval_agent(env, agent, games=500)

In [ ]:
torch.save(agent.model.state_dict(), 'agent_weights.pt')